### **Transactions : -**

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
import dask.dataframe as dd

df_t = dd.read_parquet("transactions/batch-*/*.parquet")
df_ta= dd.read_parquet("transactions_additional/batch-*/*.parquet")
df_train = dd.read_parquet("train_labels.parquet")
df_acc = dd.read_parquet("accounts.parquet")
df_time = dd.read_parquet("account_age.parquet")
df_acc.head()

,account_id,account_status,product_code,currency_code,account_opening_date,branch_code,branch_pin,avg_balance,product_family,nomination_flag,cheque_allowed,cheque_availed,num_chequebooks,last_mobile_update_date,kyc_compliant,last_kyc_date,rural_branch,monthly_avg_balance,quarterly_avg_balance,daily_avg_balance,freeze_date,unfreeze_date
0,ACCT_000000,active,200,1,2023-08-30,7982,515008.0,53720.31,K,Y,Y,Y,0,<NA>,Y,2023-11-04,Y,50111.60,55554.79,66771.57,<NA>,<NA>
1,ACCT_000002,active,1133,1,2019-08-05,5479,144074.0,9492.82,O,N,N,N,0,<NA>,Y,2021-10-03,N,9169.02,6925.45,8737.84,<NA>,<NA>
2,ACCT_000003,active,146,1,2023-10-31,8227,190033.0,93286.70,S,N,Y,Y,3,<NA>,Y,2024-10-25,N,86326.00,100171.75,78181.17,<NA>,<NA>
3,ACCT_000004,active,191,1,2017-11-25,3285,400035.0,555715.54,K,Y,Y,N,0,<NA>,Y,2023-10-29,N,647193.23,509836.62,699191.21,<NA>,<NA>
4,ACCT_000005,active,200,1,2019-01-03,2081,160035.0,16380.55,K,N,Y,N,0,<NA>,Y,2021-12-07,N,15878.60,13651.60,21249.07,<NA>,<NA>


In [ ]:
df_main = df_main.merge(
    df_other[['account_id', 'days_since_account_open']],
    on='account_id',
    how='left'
)

In [4]:
print(df_acc.shape[0].compute())

160153


In [6]:
import duckdb

# Connect to DuckDB
conn = duckdb.connect()

# Execute query reading directly from parquet files and save result
conn.execute("""
COPY (
    WITH last_transaction AS (
        SELECT 
            account_id,
            MAX(transaction_timestamp::DATE) as last_transaction_date
        FROM read_parquet('transactions/batch-*/*.parquet')
        GROUP BY account_id
    )
    SELECT 
        a.account_id,
        a.account_opening_date::DATE as account_opening_date,
        lt.last_transaction_date,
        DATEDIFF('day', a.account_opening_date::DATE, lt.last_transaction_date) as days_since_account_open
    FROM read_parquet('accounts.parquet') a
    LEFT JOIN last_transaction lt ON a.account_id = lt.account_id
) TO 'account_age.parquet' (FORMAT PARQUET)
""")

# Close the connection
conn.close()

print("Account age calculation completed and saved to account_age.parquet")

Account age calculation completed and saved to account_age.parquet


In [7]:
df = dd.read_parquet("account_age.parquet")
print(df.shape[0].compute())
df.head()

160153


,account_id,account_opening_date,last_transaction_date,days_since_account_open
0,ACCT_002612,2023-05-08,2025-06-29,783.0
1,ACCT_002618,2023-02-01,2025-06-28,878.0
2,ACCT_008260,2023-01-11,2025-06-28,899.0
3,ACCT_008275,2023-01-25,2025-06-28,885.0
4,ACCT_008309,2024-01-31,2025-06-28,514.0


In [5]:
import duckdb

con = duckdb.connect()

result = con.execute("""
SELECT
    COUNT(*) AS total_transactions,
    COUNT(tl.account_id) AS matched_accounts,
    COUNT(*) - COUNT(tl.account_id) AS unmatched_accounts
FROM read_parquet('transactions_additional/batch-*/*.parquet') ta
LEFT JOIN read_parquet('train_labels.parquet') tl
ON ta.account_id = tl.account_id
""").df()

print(result)

BinderException: Binder Error: Table "ta" does not have a column named "account_id"

Candidate bindings: : "latitude"

LINE 8: ON ta.account_id = tl.account_id
           ^

In [5]:
unique_accounts = df_t["account_id"].nunique().compute()

print("Unique account IDs:", unique_accounts)

Unique account IDs: 159776


In [6]:
df_t.head()

,transaction_id,account_id,transaction_timestamp,mcc_code,channel,amount,txn_type,counterparty_id
0,TXN_0001000000,ACCT_000494,2023-05-22T22:47:35,9384,UPD,299.55,C,CP_090625
1,TXN_0001000001,ACCT_000494,2023-05-22T22:53:31,2943,MAC,11191.27,D,CP_070045
2,TXN_0001000002,ACCT_000494,2023-05-23T01:40:04,5651,UPC,159.42,D,CP_029345
3,TXN_0001000003,ACCT_000494,2023-05-23T05:48:59,6101,UPD,23700.00,D,CP_038556
4,TXN_0001000004,ACCT_000494,2023-05-23T10:33:24,5651,UPD,2000.00,D,CP_000291


In [8]:
df_ta.head()

,transaction_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type
0,TXN_0000000000,END,NaN,NaN,122.175.102.100,53657.59,CI,<NA>,normal
1,TXN_0000000001,UPD,15.786843,78.077894,103.92.190.204,53479.10,CI,<NA>,normal
2,TXN_0000000002,CCL,NaN,NaN,<NA>,52479.10,CI,<NA>,normal
3,TXN_0000000003,UPC,15.781886,78.053709,122.124.163.67,59479.10,CI,<NA>,normal
4,TXN_0000000004,FTD,NaN,NaN,103.241.164.38,64179.10,CI,<NA>,normal


In [9]:
df_t.tail(5)

,transaction_id,account_id,transaction_timestamp,mcc_code,channel,amount,txn_type,counterparty_id
670491,TXN_0397875318,ACCT_127367,2024-01-19 20:20:58,9384,FTC,537.27,D,ACCT_046843
670492,TXN_0397875319,ACCT_173010,2021-10-20 00:19:04,5651,OCD,3335.05,C,ACCT_007400
670493,TXN_0397875320,ACCT_173010,2023-01-29 20:39:07,1139,UPD,3771.70,C,ACCT_143946
670494,TXN_0397875321,ACCT_173010,2025-05-28 10:58:24,5651,UPD,12508.22,D,ACCT_038360
670495,TXN_0397875322,ACCT_173010,2025-06-23 03:54:43,5651,UPC,101.23,C,ACCT_090272


In [11]:
df_t.tail(5)

,transaction_id,account_id,transaction_timestamp,mcc_code,channel,amount,txn_type,counterparty_id
670491,TXN_0397875318,ACCT_127367,2024-01-19 20:20:58,9384,FTC,537.27,D,ACCT_046843
670492,TXN_0397875319,ACCT_173010,2021-10-20 00:19:04,5651,OCD,3335.05,C,ACCT_007400
670493,TXN_0397875320,ACCT_173010,2023-01-29 20:39:07,1139,UPD,3771.70,C,ACCT_143946
670494,TXN_0397875321,ACCT_173010,2025-05-28 10:58:24,5651,UPD,12508.22,D,ACCT_038360
670495,TXN_0397875322,ACCT_173010,2025-06-23 03:54:43,5651,UPC,101.23,C,ACCT_090272


In [12]:
df_t.isnull().sum().compute()

transaction_id           0
account_id               0
transaction_timestamp    0
mcc_code                 0
channel                  0
amount                   0
txn_type                 0
counterparty_id          0
dtype: int64

In [16]:
print("transactions            :", df_t.shape[0].compute(), " | ", df_t.shape[1])
print("transactions_additional :", df_ta.shape[0].compute()," | ", df_ta.shape[1])

transactions            : 396875323  |  8
transactions_additional : 397875323  |  9


In [18]:
df_ta.transaction_id.nunique().compute()

np.int64(397875323)

In [17]:
total = df_t.shape[0].compute()
unique = df_t.transaction_id.nunique().compute()

print("Total:", total)
print("Unique:", unique)
print("Duplicates:", total - unique)

Total: 396875323
Unique: 396875323
Duplicates: 0


### Findig the missing transaction id  : - 

In [26]:
missing_in_t = df_ta.merge(
    df_t[["transaction_id"]],
    on="transaction_id",
    how="left",
    indicator=True
)

missing_in_t = missing_in_t[missing_in_t["_merge"] == "left_only"]

missing_in_t.head()

,transaction_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type,_merge
708555,TXN_0000000797,UPC,NaN,NaN,<NA>,480039.89,CI,<NA>,normal,left_only
708556,TXN_0000001623,OCD,NaN,NaN,<NA>,1192862.15,CI,<NA>,cash,left_only
708557,TXN_0000001746,IPM,15.781905,78.032778,<NA>,-9098803.15,CI,<NA>,normal,left_only
708558,TXN_0000001831,UPC,NaN,NaN,<NA>,-8427961.63,CI,<NA>,normal,left_only
708559,TXN_0000001986,UPC,15.810554,78.063804,103.45.124.134,-11530560.51,CI,<NA>,normal,left_only


### **Ready to merge : -**

In [5]:
import dask.dataframe as dd
df_t = dd.read_parquet("transactions/batch-*/*.parquet")
df_ta = dd.read_parquet("transactions_additional/batch-*/*.parquet")

In [6]:
# Merge using transaction_id
merged = df_t.merge(
    df_ta,
    on="transaction_id",
    how="left"
)

# Preview
merged.head()

,transaction_id,account_id,transaction_timestamp,mcc_code,channel,amount,txn_type,counterparty_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type
0,TXN_0091588263,ACCT_046536,2024-07-01T19:59:04,5682,UPC,100.00,C,CP_092582,UPC,NaN,NaN,106.73.5.156,-12269001.49,CI,<NA>,normal
1,TXN_0091588556,ACCT_046536,2025-01-24T07:04:16,4106,UPC,145.28,C,CP_053874,UPC,NaN,NaN,<NA>,-12019518.11,CI,<NA>,normal
2,TXN_0091588848,ACCT_046537,2021-04-22T20:52:17,2955,UPC,133000.00,C,CP_075883,UPC,NaN,NaN,182.81.122.85,-1488393.17,CI,<NA>,normal
3,TXN_0091589105,ACCT_046537,2021-09-19T19:32:58,2017,UPC,25605.04,C,CP_086754,UPC,NaN,NaN,157.8.163.100,-1936632.31,CI,<NA>,normal
4,TXN_0091589832,ACCT_046537,2023-01-26T08:42:07,2920,UPD,890600.00,C,CP_086754,UPD,NaN,NaN,<NA>,-3752519.93,CI,<NA>,normal


In [ ]:
merged.to_parquet(
    "merged_transactions/",
    write_index=False
)

In [ ]:
print("Merging_transaction             :", merged.shape[0].compute(), " | ", merged.shape[1])

In [ ]:
df_txn = dd.read_parquet('account_mcc_hour_day_avgs.parquet')
df_txn.head()

product_code, account_opening_date_x,branch_code,branch_pin,last_mobile_update_date,last_kyc_date,freeze_date,unfreeze_date,branch_address,branch_pin_code ,branch_city,branch_state,customer_id,date_of_birth,relationship_start_date,customer_pin,permanent_pin,name,gender,passbook_last_update_date,phone_number,address,account_opening_date_y,last_transaction_date